In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{Perturbations} $$

$$ dy = J \cdot dx $$
$$ dx = J^{-1} dy $$

$$ \text{Gradients} $$

$$ \nabla_x F = J^T \cdot \nabla_y F $$
$$ \nabla_y F = (J^{-1})^T \cdot \nabla_x F $$

$$ \text{Frobenius inner product properties} $$

$$ (1) \quad \left \langle A, BC \right \rangle _F = \left \langle B^T A, C \right \rangle _F $$
$$ (2) \quad \left \langle A, BC \right \rangle _F = \left \langle A C^T, B \right \rangle _F $$

## Example 1

$$ f(w) = x \cdot w $$

$$ x \in R^{n \times m}, \quad w \in R^{m \times 1}, \quad f \in R^{n \times 1} $$

$$ df = x \cdot dw $$

$$ \text{Frobenius equation} $$

$$ \left \langle \frac{d F}{dw}, dw \right \rangle _F = \left \langle \frac{dF}{df}, df \right \rangle _F = \left \langle \frac{dF}{df}, x \cdot dw \right \rangle _F = \left \langle x^T \cdot \frac{dF}{df}, dw \right \rangle _F $$

$$ \frac{d F}{dw} = x^T \cdot \frac{dF}{df} $$

In [ ]:
import torch.autograd as autograd
import torch as tr

x = tr.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=tr.float64, requires_grad=False)
w = tr.tensor([[7.0], [8.0], [9.0]], dtype=tr.float64, requires_grad=True)


class f(autograd.Function):
    @staticmethod
    def forward(ctx, w):
        ctx.save_for_backward(x)

        return x @ w

    @staticmethod
    def backward(ctx, dF_df):
        x, = ctx.saved_tensors
        
        dF_dw = x.t() @ dF_df
        return dF_dw


F = lambda w: tr.mean(f.apply(w))
F(w).backward()


def fn(w):
    return f.apply(w)


assert autograd.gradcheck(fn, (w, ), eps=1e-6, atol=1e-4, rtol=1e-4)

## Example 2

$$ f(\mathbf{w}, \mathbf{b}) = X \mathbf{w} + \mathbf{b} $$

$$ X \in R^{n \times m}, \quad \mathbf{w} \in R^{m \times 1}, \quad \mathbf{b} \in R^{n \times 1}, \quad f \in R^{n \times 1} $$

$$ df = X \, d\mathbf{w} + d\mathbf{b} $$

$$ \text{Frobenius equation} $$

$$ 
\left \langle \frac{\partial F}{\partial \mathbf{w}}, d\mathbf{w} \right \rangle _F + \left \langle \frac{\partial F}{\partial \mathbf{b}}, d\mathbf{b} \right \rangle _F =
\left \langle \frac{\partial F}{\partial f}, df \right \rangle _F =
$$

$$ 
\left \langle \frac{\partial F}{\partial f}, X \, d\mathbf{w} + d\mathbf{b} \right \rangle _F =
$$

$$ 
\left \langle \frac{\partial F}{\partial f}, X \, d\mathbf{w} \right \rangle _F + \left \langle \frac{\partial F}{\partial f}, d\mathbf{b} \right \rangle _F =
$$

$$
\left \langle X^\top \cdot \frac{\partial F}{\partial f}, d\mathbf{w} \right \rangle _F + \left \langle \frac{\partial F}{\partial f}, d\mathbf{b} \right \rangle _F \implies $$

$$ \frac{\partial F}{\partial \mathbf{w}} = X^\top \cdot \frac{\partial F}{\partial f} $$

$$ \frac{\partial F}{\partial \mathbf{b}} = \frac{\partial F}{\partial f} $$


In [ ]:
import torch.autograd as autograd
import torch as tr

x = tr.tensor([[1.0, 2.0, 3.0],
               [4.0, 5.0, 6.0]], dtype=tr.float64, requires_grad=False)

w = tr.tensor([[7.0], [8.0], [9.0]], dtype=tr.float64, requires_grad=True)
b = tr.tensor([[1.0], [2.0]], dtype=tr.float64, requires_grad=True)


class f(autograd.Function):
    @staticmethod
    def forward(ctx, w, b):
        ctx.save_for_backward(x)
        return x @ w + b

    @staticmethod
    def backward(ctx, dF_df):
        x, = ctx.saved_tensors

        dF_dw = x.t() @ dF_df
        dF_db = dF_df

        return dF_dw, dF_db


F = lambda w, b: tr.mean(f.apply(w, b))
F(w, b).backward()


def fn(w, b):
    return f.apply(w, b)


assert autograd.gradcheck(fn, (w, b), eps=1e-6, atol=1e-4, rtol=1e-4)

## Example 3

$$ f(X, Y) = X \cdot Y $$

$$ X \in R^{m \times n}, \quad Y \in R^{n \times k}, \quad f \in R^{m \times k} $$

$$ df = dX \cdot Y + X \cdot dY $$

$$ \text{Frobenius equation} $$

$$ \left \langle \frac{\partial F}{\partial  X}, dX \right \rangle _F + \left \langle \frac{\partial F}{\partial Y}, dY \right \rangle _F = \left \langle \frac{\partial F}{\partial f}, df \right \rangle _F = $$

$$ \left \langle \frac{\partial F}{\partial f}, dX \cdot Y + X \cdot dY \right \rangle _F = $$

$$ \left \langle \frac{\partial F}{\partial f}, dX \cdot Y \right \rangle _F + \left \langle \frac{\partial F}{\partial f}, X \cdot dY \right \rangle _F \overset{\text{2, 1}}{=} $$

$$ \left \langle \frac{\partial F}{\partial f} Y^\top, dX \right \rangle _F + \left \langle X^\top\frac{\partial F}{\partial f}, dY \right \rangle _F $$

$$ \frac{\partial F}{\partial X} = \frac{\partial F}{\partial f} Y^\top $$

$$ \frac{\partial F}{\partial Y} = X^\top\frac{\partial F}{\partial f} $$

In [ ]:
import torch.autograd as autograd
import torch as tr

x = tr.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=tr.float64, requires_grad=True)
y = tr.tensor([[7.0, 8.0], [9.0, 10.0], [11.0, 12.0]], dtype=tr.float64, requires_grad=True)


class f(autograd.Function):
    @staticmethod
    def forward(ctx, x, y):
        ctx.save_for_backward(x, y)
        
        return x @ y

    @staticmethod
    def backward(ctx, dF_df):
        x, y = ctx.saved_tensors

        dF_dx = dF_df @ y.t()
        dF_dy = x.t() @ dF_df
        return dF_dx, dF_dy


F = lambda x, y: tr.mean(f.apply(x, y))
F(x, y).backward()


def fn(x, y):
    return f.apply(x, y)


assert autograd.gradcheck(fn, (x, y), eps=1e-6, atol=1e-4, rtol=1e-4)